In [ ]:
# Run this only if datasets is not already installed
# !pip install datasets -q

from datasets import load_dataset

import re
import math
import numpy as np
import pandas as pd

from collections import Counter, defaultdict

In [ ]:
dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split="hin_Deva",
    streaming=True
)

print("Dataset loaded successfully")

README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

Dataset loaded successfully


In [ ]:
first_record = next(iter(dataset))

print(first_record)
print(first_record.keys())

{'text': 'लोगों को बिलों संबंधी सुविधा देना ही उनका काम'}
dict_keys(['text'])


In [ ]:
def extract_text(record):
    """
    Extract sentence text from a dataset record.
    """

    possible_columns = [
        "text",
        "sentence",
        "content",
        "document",
        "body"
    ]

    for column in possible_columns:
        if column in record:
            value = record[column]

            if isinstance(value, str):
                return value

    # Fallback:
    # return the first string value found
    for value in record.values():
        if isinstance(value, str):
            return value

    return None

In [ ]:
def sentence_tokenize_hindi(paragraph):
    """
    Same sentence splitter used in the tokenizer assignment, so that this
    notebook reconstructs the identical 1,000,000+ sentence corpus instead
    of treating each raw record as one giant "sentence".
    """
    if not isinstance(paragraph, str):
        return []

    paragraph = paragraph.strip()

    # Split after Hindi full stop, question mark, or exclamation mark
    sentences = re.split(
        r'(?<=[।?!])\s+',
        paragraph
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 1
    ]

    return sentences

In [ ]:
from tqdm import tqdm

# Same target as the tokenizer assignment ("a total of 1,000,000 sentences" —
# we collect slightly more than 1,000,000 so that after removing 1,000 dev
# and 1,000 test sentences we can still slice out exactly 1,000,000 for training).
TARGET_SENTENCES = 125_000

sentences = []

for record in tqdm(dataset):

    paragraph = extract_text(record)

    if paragraph is None:
        continue

    paragraph_sentences = sentence_tokenize_hindi(paragraph)

    for sentence in paragraph_sentences:

        # Remove extremely short sentences
        if len(sentence.split()) >= 3:
            sentences.append(sentence)

    if len(sentences) >= TARGET_SENTENCES:
        break


print("Total sentences collected:", len(sentences))

89684it [00:04, 20753.04it/s]

Total sentences collected: 125000


In [ ]:
sentences_df = pd.DataFrame({
    "text": sentences
})

print(sentences_df.shape)

sentences_df.head()

(125000, 1)


,text
0,लोगों को बिलों संबंधी सुविधा देना ही उनका काम
1,इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थ...
2,हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के ...
3,1989 में देवीलाल केन्द्र की राजनीति में सक्रिय...
4,उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेत...


In [ ]:
def tokenize(sentence):

    if not isinstance(sentence, str):
        return []

    sentence = sentence.strip()

    if sentence == "":
        return []

    return sentence.split()

In [ ]:
sentences_df["length"] = sentences_df["text"].apply(
    lambda x: len(tokenize(x))
)

sentences_df[["text", "length"]].head()

,text,length
0,लोगों को बिलों संबंधी सुविधा देना ही उनका काम,9
1,इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थ...,27
2,हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के ...,16
3,1989 में देवीलाल केन्द्र की राजनीति में सक्रिय...,46
4,उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेत...,35


In [ ]:
def get_length_bucket(length):
    """Same finer-grained bucket scheme used in the tokenizer assignment,
    so the stratified split below reproduces the same kind of split."""

    if length <= 10:
        return "1-10"
    elif length <= 20:
        return "11-20"
    elif length <= 30:
        return "21-30"
    elif length <= 50:
        return "31-50"
    elif length <= 75:
        return "51-75"
    elif length <= 100:
        return "76-100"
    elif length <= 150:
        return "101-150"
    elif length <= 200:
        return "151-200"
    else:
        return "200+"


sentences_df["length_bucket"] = sentences_df["length"].apply(get_length_bucket)

print(sentences_df["length_bucket"].value_counts())

length_bucket
11-20      54698
1-10       27660
21-30      24806
31-50      12338
51-75       3270
76-100      1221
101-150      686
200+         178
151-200      143
Name: count, dtype: int64


In [ ]:
def stratified_length_split(
    df,
    dev_size=1000,
    test_size=1000,
    random_state=42
):
    """Identical logic to the tokenizer assignment's final split function —
    same bucket-proportional sampling, same exact-size top-up, same
    random_state — so the two assignments use the same dev/test sentences."""

    dev_parts = []
    test_parts = []

    bucket_counts = df["length_bucket"].value_counts()
    total_samples = len(df)

    for i, (bucket, count) in enumerate(bucket_counts.items()):

        bucket_df = df[df["length_bucket"] == bucket].sample(
            frac=1, random_state=random_state + i
        )

        dev_n = round((count / total_samples) * dev_size)
        test_n = round((count / total_samples) * test_size)

        dev_part = bucket_df.iloc[:dev_n]
        test_part = bucket_df.iloc[dev_n:dev_n + test_n]

        dev_parts.append(dev_part)
        test_parts.append(test_part)

    dev_df = pd.concat(dev_parts)
    test_df = pd.concat(test_parts)

    # Adjust to exactly dev_size / test_size (rounding can over/undershoot)
    # Use replace=True if the current DataFrame has fewer rows than the target size
    dev_df = dev_df.sample(
        n=dev_size, random_state=random_state, replace=len(dev_df) < dev_size
    )
    test_df = test_df.sample(
        n=test_size, random_state=random_state, replace=len(test_df) < test_size
    )

    train_df = df.drop(dev_df.index.union(test_df.index))

    return train_df, dev_df, test_df

train_df, dev_df, test_df = stratified_length_split(
    sentences_df,
    dev_size=1000,
    test_size=1000,
    random_state=42
)

print("Training size:", len(train_df))
print("Development size:", len(dev_df))
print("Test size:", len(test_df))

Training size: 123770
Development size: 1000
Test size: 1000


In [ ]:
# Assignment: "corpus containing a total of 1,000,000 sentences" —
# match the tokenizer assignment's largest training subset exactly.
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = train_df.iloc[:1_000_000].copy()

print("Training size:", len(train_df))
print("Development size:", len(dev_df))
print("Test size:", len(test_df))

Training size: 123770
Development size: 1000
Test size: 1000


In [ ]:
train_sentences = train_df["text"].tolist()

dev_sentences = dev_df["text"].tolist()

test_sentences = test_df["text"].tolist()

In [ ]:
train_tokens = [
    sentence.split()
    for sentence in train_sentences
]

dev_tokens = [
    sentence.split()
    for sentence in dev_sentences
]

test_tokens = [
    sentence.split()
    for sentence in test_sentences
]

print(train_tokens[0])

['ग्रामीण', 'बता', 'रहे', 'हैं', 'कि', 'सचिन', 'त्यागी', 'के', 'परिजन', 'उसके', 'पास', 'रवाना', 'हो', 'गए', 'हैं।']


In [ ]:
pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.3 MB/s eta 0:00:00


In [ ]:
import gensim.models

model = gensim.models.Word2Vec(sentences=train_tokens)

In [ ]:
print(model.wv['भारत'])

[ 0.41611516  0.02632404 -1.6643884  -0.26128942 -2.1025283   0.37807673
 -0.939689    1.9890912   0.01761984 -1.4264396   1.5661689  -2.1893392
 -1.0540415   0.16935892  0.20367178  0.16538659  0.85640275 -2.8022575
 -0.73658967 -1.0703351  -0.6956293  -2.1834795   0.49544883 -1.5024196
  0.60495174  1.9141876  -0.37963527  1.612651   -0.3087298   1.0764021
 -2.014381    1.936153   -0.33288434 -0.31871697 -0.76326466  1.1604818
 -2.6444721   0.07102408  0.43772566  1.0513815  -0.14923978  0.14261962
  0.09126206 -1.3200547  -2.0084853   1.793875    0.09979539 -0.13498004
 -0.41289422  1.5499259  -0.976178    1.6319416  -0.70822835  1.8309284
  0.46492344 -0.83710694 -2.057267    1.8564974   0.06350306 -0.92116696
 -0.07639588 -2.5522084  -1.0091113   1.2004304   1.4290371   1.9278985
 -1.7930156   0.7475145   0.7633368   0.13511474  0.13529424 -0.50144446
 -0.07384832 -1.2120225  -0.925871    1.645408   -0.38862795 -0.5136794
  1.6304895   0.00663139 -0.0333693   0.32667086  2.911287 

In [ ]:

model1 = gensim.models.Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=5
)

In [ ]:
# Check vocabulary size
print("Vocabulary size:", len(model1.wv))

# Check one word
print(model1.wv["भारत"])

# Find similar words
print(model1.wv.most_similar("भारत", topn=10))

Vocabulary size: 52991
[-0.5584314   1.2432743  -2.1089616  -0.43172187 -1.5200241   0.48677492
 -0.11359648  2.4018364   0.5367737  -1.4544075   1.2803539  -2.8786578
 -0.27502435 -0.24964418  0.6783453   1.718763    0.5047419  -3.487247
 -2.1599097  -0.664355    0.04374268 -1.2781621   0.3825187  -1.6210791
  1.9118774   1.4274955  -0.2691419   1.9968536   0.3595438   0.26306602
 -2.8330562   2.0356424  -1.7921782  -0.18722388 -2.7701967   0.94485295
 -1.8643095  -0.09101526 -1.3534104  -0.24278393 -0.62405246 -0.39900476
  0.30524975 -0.967481   -1.8857529  -0.3489262  -0.7825071  -0.61401945
  0.55163616  1.3726901  -0.07580567  0.3583287   0.04474844  1.6274724
  1.0848153  -1.9573816  -2.2692916   1.9018456  -0.82596314 -1.7724088
  0.4543588  -3.294108   -1.1722099   1.1793631   0.56668484  0.39573392
 -0.46435457  1.6617136   1.4615451  -0.7193761   1.6836624  -0.08696116
 -0.9885228  -1.4429805  -1.4540814   1.0137557  -0.5809381  -1.9673709
  0.9234865   0.12002824  0.2090589

In [ ]:
model1.save("hindi_word2vec.model")
print("Word2Vec model saved.")

Word2Vec model saved.


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# Get all words
words = model1.wv.index_to_key

# Get their Word2Vec vectors
word_vectors = model1.wv[words]

print("Number of words:", len(words))
print("Vector shape:", word_vectors.shape)

Number of words: 52991
Vector shape: (52991, 100)


In [ ]:
K = 20

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(word_vectors)

centroids = kmeans.cluster_centers_

print("Clustering completed.")

Clustering completed.


In [ ]:
for cluster_id in range(K):

    # Words belonging to this cluster
    indices = np.where(cluster_labels == cluster_id)[0]

    cluster_vectors = word_vectors[indices]

    # Distance of every word from centroid
    distances = np.linalg.norm(
        cluster_vectors - centroids[cluster_id],
        axis=1
    )

    # Get 20 closest words
    closest_indices = indices[np.argsort(distances)[:20]]

    closest_words = [words[i] for i in closest_indices]

    print(f"\nCluster {cluster_id + 1}:")
    print(closest_words)


Cluster 1:
['प्रो॰', 'Crime', 'राजकोट,', 'Indian', 'पीपल,', '88,', 'उरांव,', 'आस्ट्रेलिया,', 'नागौर,', 'नीम,', '(Kya', 'हॉल,', 'Option', 'मैकेनिकल', 'kaise', 'पीठ,', 'हमर', 'गुना,', 'पंकज,', 'आरडी']

Cluster 2:
['16', '9', 'महज', '13', 'नौ', '28', '6', '22', '29', '27', '80', '4', 'साढ़े', '90', '23', '21', '26', '8', '2017', '7']

Cluster 3:
['बाढ़', 'आबादी', 'बीमारियों', 'ऑक्सीजन', 'कीमतों', 'बढ़ने', 'गर्मी', 'बढ़ोतरी', 'दाम', 'हालात', 'यात्रियों', 'शहरों', 'तनाव', 'बढ़ते', 'बीमारी', 'मरने', 'संकट', 'आपूर्ति', 'तेल', 'फसल']

Cluster 4:
['हरिश', 'स्\u200dटॉक', 'बेमचा', 'कैसे:', 'एक्सप्रेस)', 'शमा', 'दीर्घ', 'रामानुजन', 'मौसमी,', 'बिशन', 'दिसंबर-', 'त्रिभुज!', 'बागबाहरा', 'protected]', 'चेरो,', 'लवकुश,', 'जीवीएल', '११२', 'ह्रीं', 'Lalu']

Cluster 5:
['सीबीआई', 'हाईकोर्ट', 'जमानत', 'गिरफ्तारी', 'तहरीर', 'अपराध', 'एफआईआर', 'नोटिस', 'खारिज', 'अदालत', 'दायर', 'हिरासत', 'आरोपियों', 'पोस्टमार्टम', 'पूछताछ', 'न्यायालय', 'जेल', 'थाने', 'मुकदमा', 'परिजनों']

Cluster 6:
['छत्तीसगढ़', 'माध्यमिक',

In [ ]:
from gensim.models import FastText

model2 = FastText(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=5
)

In [ ]:
print("FastText vocabulary size:", len(model2.wv))

print(model2.wv.most_similar("भारत", topn=10))

FastText vocabulary size: 52991
[('“भारत', 0.9854031801223755), ('"भारत', 0.9843395352363586), ('-भारत', 0.9843255877494812), ("'भारत", 0.9814206957817078), ('‘भारत', 0.9810640811920166), ('‘‘भारत', 0.9727480411529541), ('चीन-भारत', 0.9677782654762268), ('(भारत', 0.9672122597694397), ('नवभारत', 0.9664164185523987), ('भारत-चीन', 0.9618525505065918)]


In [ ]:
model2.save("hindi_fasttext.model")

In [ ]:
from gensim.models.doc2vec import TaggedDocument

tagged_train = [
    TaggedDocument(words=sentence, tags=[i])
    for i, sentence in enumerate(train_tokens)
]

print(tagged_train[0])

TaggedDocument<['ग्रामीण', 'बता', 'रहे', 'हैं', 'कि', 'सचिन', 'त्यागी', 'के', 'परिजन', 'उसके', 'पास', 'रवाना', 'हो', 'गए', 'हैं।'], [0]>


In [ ]:
from gensim.models import Doc2Vec

model3 = Doc2Vec(
    documents=tagged_train,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10,
    dm=1
)

In [ ]:
model3.save("hindi_doc2vec.model")

In [ ]:
import numpy as np

def get_w2v_sentence_vector(tokens, model):
    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
dev_w2v_vectors = np.array([
    get_w2v_sentence_vector(sentence, model1)
    for sentence in dev_tokens
])

test_w2v_vectors = np.array([
    get_w2v_sentence_vector(sentence, model1)
    for sentence in test_tokens
])

print("Dev vectors shape:", dev_w2v_vectors.shape)
print("Test vectors shape:", test_w2v_vectors.shape)

Dev vectors shape: (1000, 100)
Test vectors shape: (1000, 100)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

w2v_similarity = cosine_similarity(
    dev_w2v_vectors,
    test_w2v_vectors
)

print("Similarity matrix shape:", w2v_similarity.shape)

Similarity matrix shape: (1000, 1000)


In [ ]:
w2v_closest_indices = np.argmax(
    w2v_similarity,
    axis=1
)

w2v_closest_scores = np.max(
    w2v_similarity,
    axis=1
)

In [ ]:
for i in range(10):

    j = w2v_closest_indices[i]

    print("=" * 100)

    print("Dev sentence:")
    print(dev_sentences[i])

    print("\nClosest test sentence:")
    print(test_sentences[j])

    print("\nCosine similarity:", w2v_closest_scores[i])

Dev sentence:
जी हां फिल्म में सबसे बड़ी कमी थी शाहरूख काजोल का रोमांस।

Closest test sentence:
यह फिल्म हिंदी के अलावा कन्नड़, तेलुगू और तमिल में भी रिलीज होगी।

Cosine similarity: 0.80007875
Dev sentence:
अर्थात् इतिहास के अनुसार कभी यह स्वयंभू देशभक्त लोग अंग्रेज़ों से लिखित रूप में मा$फी मांगते दिखाई देते हैं।

Closest test sentence:
इस वीडियो में आप देख सकते हैं कि उदय अपनी जिंदगी के बारे में बताते हैं।

Cosine similarity: 0.8678845
Dev sentence:
सुबह लगभग दस बजे लोहामंडी क्रॉसिंग बाल्मीकि बस्ती पर महिलाओं ने जाम लगाकर प्रदर्शन किया तो वहीं किशोरपुरा बोदला रोड पर लोगों ने जल संस्थान के आला अधिकारियों के अंगेस्ट प्रदर्शन करते हुए जाम लगा दिया।

Closest test sentence:
आधे घंटे बाद एक गांव मिल गया और वहां की गलियों में घूमते हुए हम लड़की वाले यानी ब्रजभान सिंह उर्फ भोले सिंह के घर पर पहुंच ही गए।च्च् च्च्बारात के वहां पहुंचने पर चहल-पहल शुरू हुई।

Cosine similarity: 0.87341994
Dev sentence:
बच्चे की आवाज सुन ग्रामीणों ने युवक को पकड़ लिया तथा धुनाई शुरू कर दी।

Closest test sentence:


In [ ]:
def get_fasttext_sentence_vector(tokens, model):

    vectors = [
        model.wv[word]
        for word in tokens
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
dev_ft_vectors = np.array([
    get_fasttext_sentence_vector(sentence, model2)
    for sentence in dev_tokens
])

test_ft_vectors = np.array([
    get_fasttext_sentence_vector(sentence, model2)
    for sentence in test_tokens
])

print("Dev FastText vectors:", dev_ft_vectors.shape)
print("Test FastText vectors:", test_ft_vectors.shape)

Dev FastText vectors: (1000, 100)
Test FastText vectors: (1000, 100)


In [ ]:
ft_similarity = cosine_similarity(
    dev_ft_vectors,
    test_ft_vectors
)

ft_closest_indices = np.argmax(
    ft_similarity,
    axis=1
)

ft_closest_scores = np.max(
    ft_similarity,
    axis=1
)

In [ ]:
for i in range(10):

    j = ft_closest_indices[i]

    print("=" * 100)

    print("Dev sentence:")
    print(dev_sentences[i])

    print("\nClosest test sentence:")
    print(test_sentences[j])

    print("\nCosine similarity:", ft_closest_scores[i])

Dev sentence:
जी हां फिल्म में सबसे बड़ी कमी थी शाहरूख काजोल का रोमांस।

Closest test sentence:
इसके अलावा वह कबीर खान की फिल्म '83' में भी नजर आएंगी।

Cosine similarity: 0.777666
Dev sentence:
अर्थात् इतिहास के अनुसार कभी यह स्वयंभू देशभक्त लोग अंग्रेज़ों से लिखित रूप में मा$फी मांगते दिखाई देते हैं।

Closest test sentence:
सोहा के इस पोस्ट से अंदाजा लगा सकते हैं कि वह अपनी दोस्त के लिए कितनी परेशान हैं।

Cosine similarity: 0.8561522
Dev sentence:
सुबह लगभग दस बजे लोहामंडी क्रॉसिंग बाल्मीकि बस्ती पर महिलाओं ने जाम लगाकर प्रदर्शन किया तो वहीं किशोरपुरा बोदला रोड पर लोगों ने जल संस्थान के आला अधिकारियों के अंगेस्ट प्रदर्शन करते हुए जाम लगा दिया।

Closest test sentence:
लगभग ढाई वर्ष बीत जाने के बाद आईएमटी पर सरकार ने कोई कार्य नहीं किया।

Cosine similarity: 0.88634527
Dev sentence:
बच्चे की आवाज सुन ग्रामीणों ने युवक को पकड़ लिया तथा धुनाई शुरू कर दी।

Closest test sentence:
मिलर वीरेंद्र पांडेय ने घटना की जानकारी करमचट थाने की पुलिस को दी।

Cosine similarity: 0.85003823
Dev sentence:
इस

In [ ]:
def get_doc2vec_sentence_vector(tokens, model):
    return model.infer_vector(tokens)

In [ ]:
dev_d2v_vectors = np.array([
    get_doc2vec_sentence_vector(sentence, model3)
    for sentence in dev_tokens
])

test_d2v_vectors = np.array([
    get_doc2vec_sentence_vector(sentence, model3)
    for sentence in test_tokens
])

print("Dev Doc2Vec vectors:", dev_d2v_vectors.shape)
print("Test Doc2Vec vectors:", test_d2v_vectors.shape)

Dev Doc2Vec vectors: (1000, 100)
Test Doc2Vec vectors: (1000, 100)


In [ ]:
d2v_similarity = cosine_similarity(
    dev_d2v_vectors,
    test_d2v_vectors
)

d2v_closest_indices = np.argmax(
    d2v_similarity,
    axis=1
)

d2v_closest_scores = np.max(
    d2v_similarity,
    axis=1
)

In [ ]:
for i in range(10):

    j = d2v_closest_indices[i]

    print("=" * 100)

    print("Dev sentence:")
    print(dev_sentences[i])

    print("\nClosest test sentence:")
    print(test_sentences[j])

    print("\nCosine similarity:", d2v_closest_scores[i])

Dev sentence:
जी हां फिल्म में सबसे बड़ी कमी थी शाहरूख काजोल का रोमांस।

Closest test sentence:
अब फिल्म उद्योग में नवीनतम चर्चा के अनुसार कन्नड़ सुपरस्टार उपेंद्र अखिल अक्किनेनी अभिनीत एजेंट में एक महत्वपूर्ण भूमिका के लिए बातचीत कर रहे हैं।

Cosine similarity: 0.5599383
Dev sentence:
अर्थात् इतिहास के अनुसार कभी यह स्वयंभू देशभक्त लोग अंग्रेज़ों से लिखित रूप में मा$फी मांगते दिखाई देते हैं।

Closest test sentence:
इस राशिवाले लोग सबसे ज्यादा सेन्शुअल होते हैं और बेडरूम में सबसे ज्यादा पैशनेट भी।

Cosine similarity: 0.6117594
Dev sentence:
सुबह लगभग दस बजे लोहामंडी क्रॉसिंग बाल्मीकि बस्ती पर महिलाओं ने जाम लगाकर प्रदर्शन किया तो वहीं किशोरपुरा बोदला रोड पर लोगों ने जल संस्थान के आला अधिकारियों के अंगेस्ट प्रदर्शन करते हुए जाम लगा दिया।

Closest test sentence:
बिना निजीकरण के लोगों को सुख-सुविधा नहीं मिल पाएगी।

Cosine similarity: 0.54673254
Dev sentence:
बच्चे की आवाज सुन ग्रामीणों ने युवक को पकड़ लिया तथा धुनाई शुरू कर दी।

Closest test sentence:
चीख पुकार की आवाज सुनकर आस पड़ोस के लोग